<a href="https://colab.research.google.com/github/Haross/sql_notebook/blob/main/assignments/Spring_Festival_Coordination/Spring_Festival_Coordination_Crisis_HW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌸 The Spring Festival Coordination Crisis

<img src="https://raw.githubusercontent.com/Haross/sql_notebook/main/assignments/Spring_Festival_Coordination/Spring_festival_planning_chaos.png" width="50%">

## 🧭 Introduction

Lunaris Systems is preparing a company-wide **Spring Festival** across its offices in Madrid, Barcelona, and Valencia.

The event includes:

- food stalls  
- music stages  
- workshops  
- games  
- logistics operations  

Everything has been planned… at least on paper.



## ⚠️ The Problem

Several coordination issues have appeared in the planning data.

Some activities may not have enough staff assigned relative to their requirements.  
Some employees may have volunteered but were never assigned.  
Some employees may be scheduled for overlapping activities.  
Some activities may lack sufficient support from the teams expected to help them run.

Management needs an audit of the planning system **before the festival begins**.


## 🎯 Your Role

You are part of the internal operations audit team.

Your task is to use SQL to:

- investigate coordination issues  
- identify patterns in the planning data  
- distinguish minor issues from meaningful risks  
- build defensible SQL logic  

This is not a step-by-step exercise.

You will need to:
- explore the data
- define what “problematic” means
- encode that reasoning in SQL



## 🧠 What matters most

This assignment is not about writing the longest query.

It is about:

- choosing the right joins  
- defining meaningful conditions  
- grouping data at the correct level  
- building logic that can be defended  

Strong answers focus on **reasoning**, not just syntax.

# **SQL Environment Setup (do not edit)**

In [29]:
# @title

%%capture
!mkdir -p notebook_lib
!wget --cache=off -q -O notebook_lib/sql_runner.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner.py
!wget --cache=off -q -O notebook_lib/sql_runner_store.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner_store.py
!wget --cache=off -q -O notebook_lib/sql_runner_ui_bits.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner_ui_bits.py
!wget --cache=off -q -O notebook_lib/validators.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/validators.py

!wget --cache=off -q -O notebook_lib/cloud_submitter.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/cloud_submitter.py

!wget --cache=off -q -O spring_festival_dataset.duckdb \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/assignments/Spring_Festival_Coordination/spring_festival_dataset.duckdb

!touch notebook_lib/__init__.py
#!pip install -q duckdb

from notebook_lib.sql_runner import make_sql_runner
from notebook_lib.validators import make_df_validator_nospoilers, check_process_rules
from notebook_lib.cloud_submitter import make_cloud_run_submitter
import duckdb
import pandas as pd
from pathlib import Path


In [30]:
# @title
DB_FILE = "spring_festival_dataset.duckdb"

conn = duckdb.connect(DB_FILE)
print(f"Database ready ✅ ({DB_FILE})")

Database ready ✅ (spring_festival_dataset.duckdb)


# 🗂️ The Database

Before analyzing coordination problems, you need to understand how the planning data is structured.

The Spring Festival system is composed of several related tables.

No single table contains the full picture.

To investigate coordination issues, you will need to:

- connect employees to activities  
- compare volunteers to assignments  
- understand how activities are scheduled  
- evaluate staffing relative to requirements  
- incorporate team support information  

Careful use of joins will be essential.

---

## 📊 Overview of the Tables

### 👥 employees

Information about all Lunaris employees.

- Each employee belongs to an office and a department  
- Employees may have a manager (`manager_id`)  
- Employees can volunteer for activities and be assigned to them  

---

### 📍 locations

Physical locations where activities take place.

- Each location belongs to an office  
- Includes capacity and whether the location is indoor or outdoor  
- Activities are scheduled at specific locations  

---

### 🎪 activities

Core table describing the festival activities.

- Each activity belongs to an office  
- Scheduled at a specific location and time  
- Has a required minimum number of staff (`min_staff_required`)  

This table is central to most analyses.

---

### 🙋 employee_volunteers

Tracks which employees volunteered for which activities.

- One row = one employee volunteering for one activity  
- Includes a `preference_level` (e.g., high, medium, low)  

Important:  
Volunteering does **not** guarantee assignment.

---

### 📋 activity_assignments

Tracks which employees were assigned to activities.

- One row = one assignment  
- Includes:
  - assigned role  
  - who made the assignment  
  - assignment status (`confirmed`, `tentative`, etc.)  

Important:  
Not all assignments may represent reliable staffing.  
You may need to decide which ones to include in your analysis.

---

### 👷 teams

Defines support teams used in the festival.

- Each team has a type (e.g., logistics, food, music)  
- Teams provide structural support to activities  

---

### 🔗 activity_teams

Links teams to activities.

- One row = one team assigned to support one activity  
- Activities may have:
  - multiple teams  
  - one team  
  - or no team at all  

---

## 🔑 Key Relationships

Understanding how tables connect is critical.

Some important relationships include:

- `activities.activity_id`  
  → connects to:
  - `employee_volunteers.activity_id`
  - `activity_assignments.activity_id`
  - `activity_teams.activity_id`

- `employees.employee_id`  
  → connects to:
  - `employee_volunteers.employee_id`
  - `activity_assignments.employee_id`

- `activities.location_id`  
  → connects to:
  - `locations.location_id`

- `teams.team_id`  
  → connects to:
  - `activity_teams.team_id`

---

## ⚠️ Important Observations

As you explore the database, keep these in mind:

- Not all activities have enough staff  
- Not all volunteers are assigned  
- Not all assignments are necessarily confirmed  
- Not all activities have team support  
- Employees may appear in multiple activities  

These patterns are intentional.

They are the foundation of the coordination problems you will investigate.

---

## 🧠 Your Goal

You are not just reading tables.

You are learning how to:

- combine data across multiple sources  
- identify meaningful patterns  
- distinguish normal variation from real issues  

Understanding the structure of this database is the first step toward building a defensible analysis.

In [31]:
# @title ER Diagram
%%html
<img id="er-img" style="width:60%; max-width:100%; height:auto;"
     data-light="https://raw.githubusercontent.com/Haross/sql_notebook/main/assignments/Spring_Festival_Coordination/er_spring_festival.png"
     data-dark="https://raw.githubusercontent.com/Haross/sql_notebook/main/assignments/Spring_Festival_Coordination/er_spring_festival_black.png"
     alt="ER diagram">

<script>
  const img = document.getElementById("er-img");

  function isDarkTheme() {
    // Colab sets html[theme=dark] on the top document
    const themeAttr = document.documentElement.getAttribute("theme");
    if (themeAttr) return themeAttr === "dark";

    // fallback: OS/browser preference
    return window.matchMedia && window.matchMedia("(prefers-color-scheme: dark)").matches;
  }

  function updateImage() {
    img.src = isDarkTheme() ? img.dataset.dark : img.dataset.light;
  }

  updateImage();

  // React to Colab theme toggles (attribute changes)
  new MutationObserver(updateImage).observe(document.documentElement, {
    attributes: true,
    attributeFilter: ["theme"]
  });

  // React to OS/browser theme changes (fallback)
  if (window.matchMedia) {
    const mq = window.matchMedia("(prefers-color-scheme: dark)");
    mq.addEventListener?.("change", updateImage);
    mq.addListener?.(updateImage); // older browsers
  }
</script>

# 🔎 Section 1 — Understanding the Planning System

<img src="https://raw.githubusercontent.com/Haross/sql_notebook/main/assignments/Spring_Festival_Coordination/Festival_planning_team collaboration_session_1.png" width="50%">

Before investigating coordination problems, you need to understand how the system is structured.

In this section, your goal is simple:

> Understand what data is available and how it connects.

Avoid jumping to conclusions.

As you work through the tables, try to understand:

- how employees are organized across offices  
- how activities are defined and scheduled  
- how volunteer interest is recorded  
- how assignments are made  
- how teams support activities  

You will later decide which of these elements matter most.

Strong investigations begin with structure, not conclusions.

In [32]:
# @title 1.1 — Activities

make_sql_runner(
    conn,
    runner_id="sec1_activities",
    description_md="""
## 🎪 1.1 Activities

Explore the **activities** table.

As you investigate, consider:

- How activities differ by type (food, music, workshop, etc.)
- How they are distributed across offices
- How time is represented (start and end times)
- What `min_staff_required` tells you about each activity

Pay close attention to:

- which activities require more staffing
- how activity schedules might interact with each other

You will use this information later when evaluating coordination issues.
""",
)

In [33]:
# @title 1.2 — Employees

make_sql_runner(
    conn,
    runner_id="sec1_employees",
    description_md="""
## 👥 1.2 Employees

Explore the **employees** table.

As you investigate, consider:

- How employees are distributed across offices
- How departments are structured
- How reporting relationships are represented (manager_id)

Think about:

- how employees might be assigned to activities
- whether organizational structure could affect coordination

At this stage, focus on understanding the workforce structure.
""",
)

In [34]:
# @title 1.3 — Volunteers

make_sql_runner(
    conn,
    runner_id="sec1_volunteers",
    description_md="""
## 🙋 1.3 Volunteers

Explore the **employee_volunteers** table.

This table represents employee interest in activities.

As you investigate, consider:

- How many employees volunteer for each activity
- Whether some activities attract more interest than others
- What `preference_level` might indicate

Think about:

- how volunteer interest might relate to actual assignments
- whether interest alone guarantees staffing

Do not draw conclusions yet — just understand the patterns.
""",
)

In [35]:
# @title 1.4 — Assignments

make_sql_runner(
    conn,
    runner_id="sec1_assignments",
    description_md="""
## 📋 1.4 Assignments

Explore the **activity_assignments** table.

This table represents actual staffing decisions.

As you investigate, consider:

- How employees are assigned to activities
- What roles they are assigned
- What `assignment_status` represents

Important:

Not all assignments may carry the same weight.

Think about:

- which assignments represent reliable staffing
- whether status should affect how you count staff

You will need to make decisions about this later.
""",
)

In [36]:
# @title 1.5 — Teams and Support

make_sql_runner(
    conn,
    runner_id="sec1_teams",
    description_md="""
## 🛠️ 1.5 Teams and Support

Explore the **teams** and **activity_teams** tables.

These tables represent structural support for activities.

As you investigate, consider:

- Which teams exist and what they specialize in
- How teams are assigned to activities
- Whether all activities receive team support

Think about:

- what role teams play in making an activity successful
- whether lack of support might matter in some cases

You will later decide how important this signal is.
""",
)

## CASE-WHEN: Creating Conditional Logic in SQL


Sometimes we want to create new columns based on conditions.

In SQL, we use `CASE WHEN` for this.

### Basic structure

```sql
CASE
    WHEN condition THEN value
    WHEN condition THEN value
    ELSE value
END
````

This works like an `IF / ELSE IF / ELSE` statement.


### Example

```sql
SELECT
    activity_name,
    min_staff_required,
    CASE
        WHEN min_staff_required >= 5 THEN 'high demand'
        WHEN min_staff_required >= 3 THEN 'medium demand'
        ELSE 'low demand'
    END AS demand_level
FROM activities;
```


### 🔍 What is happening here?

For each activity:

* If `min_staff_required >= 5` → label it **'high demand'**
* Else if `min_staff_required >= 3` → label it **'medium demand'**
* Otherwise → label it **'low demand'**

So this query is **creating a new column (`demand_level`)** that classifies activities based on how many staff they require.

👉 Important:
SQL evaluates the conditions **from top to bottom**, and stops at the first match.




### ⚠️ Important Note

`CASE WHEN` is a useful tool for creating conditional logic, but it is **not required** to solve the problems in this assignment.

All tasks can be solved using the SQL concepts covered in class, including:
- joins
- filtering (`WHERE`)
- grouping and aggregation
- subqueries

SQL is flexible, and there are often multiple correct ways to solve the same problem.

You are encouraged to use the approach that best reflects your understanding.

In [37]:
# @title Practice Case - When 1
make_sql_runner(
    conn,
    runner_id="case_when_practice_1",
    description_md="""
## 🧪 Practice Exercise 1

Create a query that shows:

- activity_name
- min_staff_required
- a new column called `readiness_flag` defined as:

    - 'understaffed' if `min_staff_required >= 4`
    - 'ok' otherwise

This is a simplified way to flag activities that may require more attention.

""",
    sol_sql="""
SELECT
    activity_name,
    min_staff_required,
    CASE
        WHEN min_staff_required >= 4 THEN 'understaffed'
        ELSE 'ok'
    END AS readiness_flag
FROM activities;
    """
)

In [38]:
# @title Practice Case - When 2
make_sql_runner(
    conn,
    runner_id="case_when_practice_1",
    description_md="""
## 🧪 Practice Exercise 2

Create a query that shows:

* activity_name
* assignment_status
* a new column called `status_flag`:

  * 'valid' if assignment_status = 'confirmed'
  * 'pending' otherwise

You will need to join:

* activities
* activity_assignments


""",
sol_sql="""
SELECT
    a.activity_name,
    aa.assignment_status,
    CASE
        WHEN aa.assignment_status = 'confirmed' THEN 'valid'
        ELSE 'pending'
    END AS status_flag
FROM activities a
JOIN activity_assignments aa
    ON a.activity_id = aa.activity_id;
"""
)

## End of Section 1 — Structural Understanding

At this point, you should understand:

- how the main tables relate to each other  
- how staffing, volunteering, and assignments differ  
- how time and scheduling are represented  
- how support structures are assigned  

You are **not solving the problem yet**.

You are building the foundation needed to evaluate coordination issues.

In the next section, you will begin to define what “problematic” actually means and encode that logic in SQL.

# 🧠 Section 2 — Investigating Coordination Problems

<img src="https://raw.githubusercontent.com/Haross/sql_notebook/main/assignments/Spring_Festival_Coordination/section_2_banner.png" width="50%">

By this stage, you should understand how the festival planning system is structured.

Now the audit begins.

Operations has asked you to investigate several possible coordination problems in the system.

Some issues may be minor.  
Others may point to activities that are genuinely at risk.

Your task in this section is to write SQL queries that help identify and evaluate these problems.

## Important

These are **not recipe-style exercises**.

For each statement below, you must decide:

- which tables to use  
- how to join them  
- what level of grouping makes sense  
- which conditions meaningfully identify the problem  
- how to distinguish weak signals from stronger ones  

There is not always one single correct query.

A strong answer will encode a **clear and defensible definition** of the coordination problem described.

---

## 📝 What to submit for each statement

For each of the five statements below, submit:

1. **One SQL query**
2. **A short explanation** describing:
   - what pattern your query is trying to detect
   - why your conditions are reasonable
   - why the result would matter in a coordination audit
3. Additional Investigation Notes (Optional). You may include any additional observations, alternative tests, or intermediate analyses that strengthen and support the logic of your final query.

Your grade will depend not only on whether the query runs, but on how well your SQL logic captures the coordination problem.

---

## ⚠️ A note on interpretation

Not all coordination issues carry the same weight.

In this dataset, some activities show only a **small shortfall** or a **single weak signal**.  
Others combine multiple issues in a way that suggests more serious operational risk.

As you work:

- avoid relying on a single metric in isolation  
- look for patterns where **multiple signals reinforce each other**  
- be cautious about over-interpreting weak or borderline cases  

A strong investigator looks for patterns that are:
- logically meaningful
- supported by the data
- defensible under review


In [39]:
# @title Statement 1 — Operational Readiness

make_sql_runner(
    conn,
    runner_id="statement_1",
    description_md="""
## Statement 1 — Operational Readiness

Write one SQL query to identify activities that may **not be operationally ready due to insufficient staffing**.

Focus on staffing relative to `min_staff_required`, not just whether staff exist.

Consider:
- how to count assigned staff
- how to compare against required staff
- when a staffing gap becomes meaningful
""",
)

In [40]:
# @title Statement 2 — Volunteer Utilization

make_sql_runner(
    conn,
    runner_id="statement_2",
    description_md="""
## Statement 2 — Volunteer Utilization

Write one SQL query to identify activities where **volunteer interest is not effectively used**.

Compare how many employees volunteered vs how many were actually assigned.

Consider:
- how to count volunteers
- how to compare volunteers to assignments
- when a mismatch becomes meaningful
""",
)

In [41]:
# @title Statement 3 — Coordination Conflicts

make_sql_runner(
    conn,
    runner_id="statement_3",
    description_md="""
## Statement 3 — Coordination Conflicts

Write one SQL query to identify employees assigned to **overlapping activities**.

These situations may create coordination conflicts.

You will likely need to compare multiple assignments for the same employee.

Consider:
- how to detect time overlap between activities
- how to compare assignments for the same employee
- how to avoid duplicate counting
""",
)

In [42]:
# @title Statement 4 — Weak Team Support

make_sql_runner(
    conn,
    runner_id="statement_4",
    description_md="""
## Statement 4 —  Weak Team Support

Write one SQL query to identify activities with **weak team support**.

In this dataset, teams provide structural support to activities.
Activities with limited or missing team coverage may be harder to coordinate effectively.

Consider:
- how to measure team coverage per activity
- how to handle activities with no associated teams
- whether weak team support alone is enough to raise concern

Your query should identify activities where team support may be insufficient.
""",
)

In [43]:
# @title Statement 5 — Final Coordination Risk Query

make_sql_runner(
    conn,
    runner_id="statement_5",
    description_md="""
## Statement 5 — Final Coordination Risk Query

Write one SQL query to identify the activities you consider **most at risk overall**.

Combine multiple signals such as:
- staffing gaps
- unused volunteers
- overlapping assignments
- weak team support

Return exactly:
- activity_id
- activity_name
- office
- assigned_staff_count
- min_staff_required
- unassigned_volunteers
- overlapping_assignment_conflicts
- team_count

Sort so the most concerning activities appear first.
""",
)